# LLMOps 실습 과제

`llmops_core.py`(따라하기 코드를 모은 파일)를 불러와서 진행한다. 위에서부터 순서대로 풀고, 각 과제의 **확인** 셀이 `✅`를 출력하면 통과다.

## [준비] 준비 — 실습용 프로젝트 만들기

따라하기와 섞이지 않도록 `practice/` 폴더에 새 프로젝트를 만든다. 결과를 모두 같게 재현하려고 과제 1~6은 **BERT와 GPT를 끈 상태**(TF-IDF + mock)로 진행한다. 과제 7에서 켜 본다.

In [ ]:
import llmops_core as lc

lc.set_project("practice")          # 실습 전용 폴더
lc.CONFIG["use_bert"] = False
lc.CONFIG["use_llm"] = False
lc.init_project(reset=True)
lc.check_and_run()

## [기본] 새 FAQ를 넣고 자동 반영 확인하기

쇼핑몰이 **선물 포장** 서비스를 시작했다. FAQ `D9`를 추가하고, 고객이 물어볼 만한 질문 2개를 평가셋에 넣은 뒤 파이프라인을 실행하라. 새 버전이 활성화되고 챗봇이 D9로 답하면 성공이다.

- 제목: `선물 포장이 되나요?`
- 답변: `주문서에서 선물 포장을 선택하면 2000원에 포장해 드립니다. 메시지 카드는 무료입니다.`
- 평가 질문 예: `선물용으로 포장 가능해요?`, `포장해서 보내줄 수 있어요?`

In [ ]:
# TODO 1: lc.add_faq(...)로 D9 추가

# TODO 2: lc.add_eval(...)로 평가 질문 2개 추가 (정답 D9)

# TODO 3: 변경 감지 → 파이프라인 실행


In [ ]:
# 확인
r = lc.ask("선물 포장 얼마예요?")
assert lc.load_registry()["active"] != "v001", "새 버전이 활성화되지 않았습니다"
assert "[D9]" in r["answer"], "D9로 답하지 않았습니다"
print("✅ 과제 1 통과")

## [기본] 틀린 질문 고치기 — 사전으로 안 되면 FAQ를 보강한다

고객이 `공짜 배송 기준이 뭐예요?`라고 물었다. 정답은 D2(배송비)다.

1. 평가셋에 넣고 실행해 **틀리는 것**을 확인한다.
2. 동의어 사전에 `공짜 → 무료`를 넣고 다시 실행한다. 여전히 틀린다.
3. 원인 분석 셀로 어떤 단어 때문에 D1이 이기는지 확인한다.
4. D2의 **제목을 보강**해(`lc.edit_faq`) 통과시킨다.

In [ ]:
# 1) 문제 재현
lc.add_eval("공짜 배송 기준이 뭐예요?", "D2")
lc.check_and_run()

# 2) TODO: 동의어 사전에 '공짜' → '무료' 추가 후 실행


# 3) 원인 분석 — 질문과 겹치는 단어를 문서별로 본다
idx = lc.load_index(sorted(lc.load_registry()["versions"])[-1], lc.CONFIG)   # 가장 최근 버전
q = lc.clean_text("공짜 배송 기준이 뭐예요?", idx["synonyms"])
print("질문 토큰:", lc.tokenize_ko(q))
for d, c in zip(idx["docs"], idx["corpus"]):
    print(d["id"], [t for t in lc.tokenize_ko(c) if t in lc.tokenize_ko(q)])

# 4) TODO: lc.edit_faq("D2", title=...)로 D2 제목을 보강하고 실행


In [ ]:
# 확인
m = lc.manifest_of(lc.load_registry()["active"])
row = [r for r in m["rows"] if r["q"] == "공짜 배송 기준이 뭐예요?"][0]
assert row["hit1"], f"아직 틀립니다: {row['top']}"
print("✅ 과제 2 통과")

## [기본] 단위 정제 — 미터를 센티미터로 환산하기

FAQ D7에는 `가로 150cm 이상`이라고 적혀 있는데 고객은 `2미터 책장도 화물로 와요?`라고 묻는다. 현재 정제는 `2미터 → 2m`까지만 바꾼다. `m`를 `cm`로 **숫자까지 환산**하는 규칙을 `lc.UNIT_RULES`에 추가하라.

- 힌트: 바꿀 값에 함수(`lambda m: ...`)를 쓰면 숫자를 계산할 수 있다. `12mm`, `150cm`, `5km`는 건드리면 안 된다.
- 생각해 보기: 규칙은 **코드**라서 `check_and_run()`이 변경을 감지하지 못한다. 실제 서비스에서는 무엇이 이 역할을 할까?

In [ ]:
# 1) 문제 재현
lc.add_eval("2미터 책장도 화물로 와요?", "D7")
lc.check_and_run()

# 2) TODO: m → cm 환산 규칙을 lc.UNIT_RULES에 append


print(lc.clean_text("2미터 책장, 1.5m 소파, 12mm 볼트, 150cm, 5km"))

# 3) 코드가 바뀐 것은 감지되지 않는다 → 강제 실행
lc.check_and_run()
lc.check_and_run(force=True)

In [ ]:
# 확인
assert lc.clean_text("2미터 책장, 1.5m 소파, 12mm 볼트, 150cm, 5km") == "200cm 책장, 150cm 소파, 12mm 볼트, 150cm, 5km"
m = lc.manifest_of(lc.load_registry()["active"])
assert all(r["hit1"] for r in m["rows"] if r["gold"] == "D7"), "D7 질문이 아직 틀립니다"
print("✅ 과제 3 통과")

## [응용] 데이터 검증 규칙 추가하기

운영자가 실수로 너무 짧은 답변이나 **이미 있는 제목**으로 FAQ를 올리는 일이 생긴다. 두 가지 검사 함수를 만들어 `lc.CUSTOM_CHECKS`에 넣어라. 검사 함수는 `(docs, evals)`를 받아 **오류 메시지 리스트**를 돌려준다.

- `check_answer_length`: 답변이 15자 미만이면 오류
- `check_duplicate_title`: 제목이 같은 FAQ가 2개 이상이면 오류

In [ ]:
def check_answer_length(docs, evals):
    # TODO
    return []

def check_duplicate_title(docs, evals):
    # TODO
    return []

lc.CUSTOM_CHECKS += [check_answer_length, check_duplicate_title]

# 시험 — 둘 다 '검증 실패'가 나와야 한다
lc.add_faq("D10", "쿠폰은 어디서 받나요?", "이벤트 페이지")
r1 = lc.check_and_run()
lc.remove_faq("D10")
lc.add_faq("D11", "교환·반품은 언제까지 되나요?", "교환과 반품 기간은 상품 상세 페이지의 안내를 따릅니다.")
r2 = lc.check_and_run()
lc.remove_faq("D11")
lc.check_and_run()

In [ ]:
# 확인
assert r1 and r1["status"] == "검증 실패", "짧은 답변이 걸러지지 않았습니다"
assert r2 and r2["status"] == "검증 실패", "중복 제목이 걸러지지 않았습니다"
print("✅ 과제 4 통과")

## [응용] 게이트 강화 — 꼭 맞혀야 하는 질문

지금 게이트는 **새로 추가한 질문이 틀려도** 전체 점수만 넘으면 통과시킨다 (과제 2·3에서 확인). 평가셋에 `must_pass: true`로 표시한 질문은 반드시 1위로 맞혀야 통과하도록 게이트 함수를 만들어 `lc.CUSTOM_GATES`에 넣어라. 게이트 함수는 `result`를 받아 `(통과 여부, 메시지)`를 돌려준다. `result["rows"]`의 각 행에 `must_pass`, `hit1`이 있다.

그다음 필수 질문 `주문한 거 언제 받아보죠?`(정답 D1)가 **보류**되는 것을 확인하고, FAQ를 고쳐 통과시켜라.

In [ ]:
def gate_must_pass(result):
    # TODO: must_pass인데 hit1이 False인 질문을 찾는다
    return True, ""

lc.CUSTOM_GATES.append(gate_must_pass)

lc.add_synonym(r"받아\s?[보볼봐]\w*", "도착", "따라하기와 같은 사전")
lc.add_eval("주문한 거 언제 받아보죠?", "D1", must_pass=True)
r1 = lc.check_and_run()      # '보류'가 나와야 한다

# TODO: D1을 보강해 통과시키기 (힌트: 질문의 '도착'이 D1 제목에도 있으면?)


In [ ]:
# 확인
ok, _ = gate_must_pass({"rows": [{"q": "시험", "must_pass": True, "hit1": False}]})
assert not ok, "gate_must_pass가 틀린 필수 질문을 잡지 못합니다"
assert r1 and r1["status"] == "보류", "필수 질문이 틀렸는데 게이트를 통과했습니다"
m = lc.manifest_of(lc.load_registry()["active"])
must = [r for r in m["rows"] if r.get("must_pass")]
assert must and all(r["hit1"] for r in must), "필수 질문이 활성 버전에서 아직 틀립니다"
print("✅ 과제 5 통과")

## [응용] 모니터링 지표 추가하기

검색은 됐지만 점수가 낮은 답은 **조용한 오답**일 가능성이 크다. `top_score`가 0.2 미만인 요청의 비율 `low_confidence_rate`를 `lc.CUSTOM_METRICS`에 추가하고, `lc.ALERT`에 경보 기준 0.3을 넣어라. 지표 함수는 `(logs, feedbacks)`를 받아 숫자를 돌려준다.

In [ ]:
# TODO: lc.CUSTOM_METRICS["low_confidence_rate"] = ...
# TODO: lc.ALERT["low_confidence_rate"] = 0.3

for q in ["택배 언제 와요?", "오배송 왔어요", "상품이 깨져서 왔어요", "환불 며칠 걸려요?"]:
    lc.ask(q)
m = lc.monitor()

In [ ]:
# 확인
assert "low_confidence_rate" in m, "지표가 추가되지 않았습니다"
assert 0 <= m["low_confidence_rate"] <= 1
print("✅ 과제 6 통과 — 점수가 낮았던 질문:", [r["query"] for r in lc.read_jsonl(lc.LOGS / "serve_log.jsonl") if r["top_score"] < 0.2])

## [응용] 설정 실험 — 설정을 바꿔도 파이프라인이 돈다

`CONFIG`도 변경 감지 대상이다. 값을 하나씩 바꿔 `check_and_run()`을 실행하고 결과를 표에 기록하라. 바꾼 값을 원래대로 돌리면 '같은 데이터로 돌아와 실행 생략'이 나오는 것도 확인한다.

| 실험 | 바꾼 설정 | Hit@1 | 게이트 | 관찰 |
|---|---|---|---|---|
| A | `min_score` 0.05 → 0.3 | | | |
| B | `top_k` 3 → 1 | | | |
| C | `use_bert` True (torch 필요) | | | |
| D | C + `alpha` 0.3 | | | |
| E | `use_llm` True (API 키 필요) | | | 인용률 변화 |

In [ ]:
# 실험 A
lc.CONFIG["min_score"] = 0.3
lc.check_and_run()
lc.CONFIG["min_score"] = 0.05
lc.check_and_run()

# TODO: 실험 B~E도 같은 방식으로 실행하고 표를 채운다


## [도전] 도전 — 같은 LLMOps를 제조 현장 FAQ에 적용하기

코드는 그대로 두고 **데이터만 바꿔** 제조 설비 FAQ 챗봇을 만든다. `factory/` 프로젝트를 만들고 아래를 채워라.

1. 동의어 사전: 현장 약어를 표준 용어로 (`M/C→머시닝센터`, `NG→불량`, `에러→알람`, `#3→3호기`, `툴→공구`, `갈아→교체` 등)
2. 평가 질문 5개가 모두 통과하는 v001 만들기
3. 새 FAQ `F6 절삭유 보충`과 질문 `쿨런트 언제 채워요?`를 넣었을 때 틀리면, 사전을 보강해 통과시키기

In [ ]:
import json
lc.set_project("factory")
lc.write_csv(lc.FAQ_FILE, ["id", "title", "text"], [
    ("F1", "설비 알람이 뜨면 어떻게 하나요?", "설비 화면의 알람 코드를 확인하고 비상정지한 뒤 설비 담당자에게 알립니다. 임의로 재가동하지 않습니다."),
    ("F2", "불량이 나오면 어떻게 처리하나요?", "불량품은 빨간 상자에 격리하고 불량 일지에 수량과 원인을 기록합니다."),
    ("F3", "치수 공차는 얼마인가요?", "가공 치수 공차는 ±0.05mm 이내이며 벗어나면 불량으로 판정합니다."),
    ("F4", "작업 전 점검은 무엇을 하나요?", "작업 전 절삭유 수위, 공구 마모, 안전 커버 닫힘을 확인하고 점검표에 서명합니다."),
    ("F5", "머시닝센터 공구 교체 주기는?", "머시닝센터 공구는 가공 500개마다 또는 마모가 보이면 교체합니다."),
])
# TODO 1: 제조 동의어 사전 (pattern, replace, note)
lc.write_csv(lc.SYN_FILE, ["pattern", "replace", "note"], [
    # (r"\bM/C\b|엠씨", "머시닝센터", "설비 약어"),
])
lc.EVAL_FILE.write_text("".join(json.dumps({"q": q, "gold": g, "source": "초기"}, ensure_ascii=False) + "\n" for q, g in [
    ("#3 M/C 툴 언제 갈아요?", "F5"), ("NG 나오면 어디다 둬요?", "F2"), ("에러 뜨면 어떡해요?", "F1"),
    ("치수 0.05미리 넘으면 어떻게 돼요?", "F3"), ("작업 시작 전에 뭐 확인해요?", "F4")]), encoding="utf-8")
lc.check_and_run()

# TODO 2: 새 FAQ F6 추가 + 평가 질문 → 실행 → 틀리면 사전 보강 → 실행
# 답변 예: 절삭유 수위가 게이지의 MIN 선 아래로 내려가면 지정 절삭유로 MAX 선까지 보충합니다.


In [ ]:
# 확인
m = lc.manifest_of(lc.load_registry()["active"])
assert len(m["rows"]) >= 6 and all(r["hit1"] for r in m["rows"]), [r["q"] for r in m["rows"] if not r["hit1"]]
print("✅ 과제 8 통과 — 제조 FAQ 챗봇", lc.load_registry()["active"], "활성")

## [도전] 도전 — watch.py로 진짜 자동 실행 보기

노트북 대신 **파일 저장만으로** 파이프라인이 도는 것을 확인한다.

1. 터미널에서 실습 폴더로 이동해 `python watch.py`를 실행한다.
2. VS Code나 엑셀에서 `data/faq.csv`를 열어 FAQ를 한 줄 추가하고 저장한다. → 5초 안에 파이프라인 출력이 나온다.
3. 답변에 전화번호를 넣어 저장해 본다. → 검증 실패로 멈추고 활성 버전은 그대로다.
4. 그 줄을 지우고 저장한다. → '같은 데이터로 돌아와 실행 생략'.
5. 터미널 출력을 캡처해 제출한다.